# TripPulse — Week 5: Silver Candidate Transformation

**Notebook:** `notebooks/03_silver_transformations.ipynb`  
**Project:** TripPulse — Urban Mobility Analytics  
**Technology:** Databricks Free Edition · Spark SQL · Delta

## Week 5 objective

Transform the completed Week-4 Bronze records into typed, standardised and traceable **Silver Candidate** tables while preserving the approved source grain and Bronze lineage.

`Bronze → standardise → type safely → calculate approved fields → persist Candidate → validate`

**Important boundary:** Silver Candidate is structured but **not trusted**. Week 6 performs Data Quality routing to Trusted Silver or quarantine.

This notebook is project-specific.


## 1. Week-4 handoff and project-specific inputs

The attached Week-4 Bronze notebook uses:

| Bronze input | Actual Week-4 table | Grain |
|---|---|---|
| `zones.csv` | `bronze_trippulse_zones` | one zone row |
| `drivers.json` | `bronze_trippulse_drivers` | one driver row |
| `trips.parquet` | `bronze_trippulse_trips` | one trip/request row |
| `payments.csv` | `bronze_trippulse_payments` | one payment-attempt row |


In [ ]:
USE CATALOG `TripPulse`;
USE SCHEMA `default`;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;


In [ ]:
SHOW TABLES LIKE 'bronze_trippulse_*';


### Baseline counts


In [ ]:
SELECT 'zones' AS entity, COUNT(*) AS bronze_rows FROM bronze_trippulse_zones
UNION ALL
SELECT 'drivers', COUNT(*) FROM bronze_trippulse_drivers
UNION ALL
SELECT 'trips', COUNT(*) FROM bronze_trippulse_trips
UNION ALL
SELECT 'payments', COUNT(*) FROM bronze_trippulse_payments
ORDER BY entity;


## 2. Transformation contract

| Candidate table | Approved Week-5 work | Grain |
|---|---|---|
| `silver_zones_candidate` | parse date/boolean; standardise IDs and controlled categories; retain lineage | one zone |
| `silver_drivers_candidate` | parse date/timestamp/rating/integer; standardise vehicle/service/status; retain lineage | one driver |
| `silver_trips_candidate` | parse lifecycle timestamps/decimals; standardise identifiers/status/service; derive approved fields; retain lineage | one trip |
| `silver_payments_candidate` | parse attempt/time/amount/final flag; standardise method/status/reason; retain lineage; preserve attempt grain | one payment attempt |

## 3. Source-specific timestamp note

The attached Week-4 notebook reads the TripPulse Parquet lifecycle timestamps as `LongType` because the source stores them as Parquet `INT64 TIMESTAMP(NANOS)`. Therefore the Trip Candidate conversion must turn those epoch-nanosecond values into Spark `TIMESTAMP` values before calculations.

This is a **technical source-format conversion**, not a business correction. The conversion below uses `timestamp_micros(nanoseconds / 1000)` so the instant represented by the source value is preserved.


# 4. Zones Candidate

**Bronze:** `bronze_trippulse_zones`  
**Candidate:** `silver_zones_candidate`  
**Grain:** one zone row

### Approved transformations
- identifiers: `TRIM` + approved casing
- controlled categories: lowercase
- `is_active`: typed BOOLEAN
- `effective_from`: typed DATE
- `zone_name`: trim only; no invented category/case correction
- Bronze lineage is retained


In [ ]:
CREATE OR REPLACE TEMP VIEW zones_standardised AS
SELECT
    UPPER(TRIM(zone_id)) AS zone_id,
    TRIM(zone_name) AS zone_name,
    LOWER(TRIM(zone_type)) AS zone_type,
    UPPER(TRIM(city_code)) AS city_code,
    LOWER(TRIM(demand_band)) AS demand_band,
    is_active,
    effective_from,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version,
    _record_hash
FROM bronze_trippulse_zones;


In [ ]:
CREATE OR REPLACE TEMP VIEW zones_typed AS
SELECT
    zone_id,
    zone_name,
    zone_type,
    city_code,
    demand_band,
    TRY_CAST(is_active AS BOOLEAN) AS is_active,
    TRY_CAST(effective_from AS DATE) AS effective_from,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version AS _bronze_schema_version,
    _record_hash AS _bronze_record_hash
FROM zones_standardised;


In [ ]:
DESCRIBE zones_typed;


In [ ]:
CREATE OR REPLACE TABLE silver_zones_candidate
USING DELTA
AS
SELECT
    *,
    CURRENT_TIMESTAMP() AS _candidate_created_at,
    'trippulse_silver_candidate_v1.0' AS _candidate_schema_version
FROM zones_typed;


In [ ]:
SELECT zone_id, zone_name, zone_type, city_code, demand_band,
       is_active, effective_from,
       _source_file_name, _bronze_record_hash
FROM silver_zones_candidate
LIMIT 10;


# 5. Drivers Candidate

**Bronze:** `bronze_trippulse_drivers`  
**Candidate:** `silver_drivers_candidate`  
**Grain:** one driver snapshot

### Approved transformations
- IDs: uppercase + trim
- vehicle/service/status categories: lowercase + trim
- `onboard_date`: DATE
- `last_status_update_ts`: TIMESTAMP
- `rating`: DECIMAL(3,2)
- `lifetime_completed_trips`: INT
- `source_record_version`: INT
- no reference join is performed in Week 5


In [ ]:
CREATE OR REPLACE TEMP VIEW drivers_standardised AS
SELECT
    UPPER(TRIM(driver_id)) AS driver_id,
    UPPER(TRIM(home_zone_id)) AS home_zone_id,
    onboard_date,
    LOWER(TRIM(vehicle_type)) AS vehicle_type,
    LOWER(TRIM(service_type)) AS service_type,
    LOWER(TRIM(driver_status)) AS driver_status,
    rating,
    lifetime_completed_trips,
    last_status_update_ts,
    source_record_version,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version,
    _record_hash
FROM bronze_trippulse_drivers;


In [ ]:
CREATE OR REPLACE TEMP VIEW drivers_typed AS
SELECT
    driver_id,
    home_zone_id,
    TRY_CAST(onboard_date AS DATE) AS onboard_date,
    vehicle_type,
    service_type,
    driver_status,
    TRY_CAST(rating AS DECIMAL(3,2)) AS rating,
    TRY_CAST(lifetime_completed_trips AS INT) AS lifetime_completed_trips,
    TRY_CAST(last_status_update_ts AS TIMESTAMP) AS last_status_update_ts,
    TRY_CAST(source_record_version AS INT) AS source_record_version,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version AS _bronze_schema_version,
    _record_hash AS _bronze_record_hash
FROM drivers_standardised;


In [ ]:
DESCRIBE drivers_typed;


In [ ]:
CREATE OR REPLACE TABLE silver_drivers_candidate
USING DELTA
AS
SELECT
    *,
    CURRENT_TIMESTAMP() AS _candidate_created_at,
    'trippulse_silver_candidate_v1.0' AS _candidate_schema_version
FROM drivers_typed;


In [ ]:
SELECT driver_id, home_zone_id, onboard_date, vehicle_type,
       service_type, driver_status, rating, lifetime_completed_trips,
       last_status_update_ts, source_record_version,
       _source_file_name, _bronze_record_hash
FROM silver_drivers_candidate
LIMIT 10;


# 6. Trips Candidate

**Bronze:** `bronze_trippulse_trips`  
**Candidate:** `silver_trips_candidate`  
**Grain:** one ride request/trip

### Approved trip derivations

| Field | Definition / eligibility |
|---|---|
| `response_seconds` | accepted timestamp − request timestamp when both parse and accept is not before request |
| `wait_seconds` | pickup timestamp − driver-accept timestamp when both parse and pickup is not before accept |
| `trip_duration_seconds` | dropoff timestamp − pickup timestamp for locally eligible completed records |
| `is_completed` | `trip_status = completed` |
| `is_cancelled` | status is `cancelled_by_rider` or `cancelled_by_driver` |
| `is_unfulfilled` | `trip_status = unfulfilled` |
| `is_surge_trip` | valid surge multiplier > 1.00 |
| `distance_variance_km` | actual distance − estimated distance when both are present |
| `fare_variance_inr` | final fare − estimated fare when the completed lifecycle and both fare values are eligible |

No default values are invented when the required inputs are missing or not locally eligible.


In [ ]:
CREATE OR REPLACE TEMP VIEW trips_standardised AS
SELECT
    UPPER(TRIM(trip_id)) AS trip_id,
    request_ts,
    driver_accept_ts,
    pickup_ts,
    dropoff_ts,
    cancel_ts,
    UPPER(TRIM(driver_id)) AS driver_id,
    UPPER(TRIM(pickup_zone_id)) AS pickup_zone_id,
    UPPER(TRIM(dropoff_zone_id)) AS dropoff_zone_id,
    LOWER(TRIM(service_type)) AS service_type,
    LOWER(TRIM(trip_status)) AS trip_status,
    LOWER(TRIM(cancellation_reason)) AS cancellation_reason,
    estimated_distance_km,
    actual_distance_km,
    estimated_fare_inr,
    final_fare_inr,
    surge_multiplier,
    record_created_ts,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version,
    _record_hash
FROM bronze_trippulse_trips;


In [ ]:
CREATE OR REPLACE TEMP VIEW trips_typed AS
SELECT
    trip_id,

    CASE WHEN request_ts IS NOT NULL
         THEN timestamp_micros(TRY_CAST(request_ts / 1000 AS BIGINT))
    END AS request_ts,

    CASE WHEN driver_accept_ts IS NOT NULL
         THEN timestamp_micros(TRY_CAST(driver_accept_ts / 1000 AS BIGINT))
    END AS driver_accept_ts,

    CASE WHEN pickup_ts IS NOT NULL
         THEN timestamp_micros(TRY_CAST(pickup_ts / 1000 AS BIGINT))
    END AS pickup_ts,

    CASE WHEN dropoff_ts IS NOT NULL
         THEN timestamp_micros(TRY_CAST(dropoff_ts / 1000 AS BIGINT))
    END AS dropoff_ts,

    CASE WHEN cancel_ts IS NOT NULL
         THEN timestamp_micros(TRY_CAST(cancel_ts / 1000 AS BIGINT))
    END AS cancel_ts,

    driver_id,
    pickup_zone_id,
    dropoff_zone_id,
    service_type,
    trip_status,
    cancellation_reason,

    TRY_CAST(estimated_distance_km AS DECIMAL(7,2)) AS estimated_distance_km,
    TRY_CAST(actual_distance_km AS DECIMAL(7,2)) AS actual_distance_km,
    TRY_CAST(estimated_fare_inr AS DECIMAL(10,2)) AS estimated_fare_inr,
    TRY_CAST(final_fare_inr AS DECIMAL(10,2)) AS final_fare_inr,
    TRY_CAST(surge_multiplier AS DECIMAL(4,2)) AS surge_multiplier,

    CASE WHEN record_created_ts IS NOT NULL
         THEN timestamp_micros(TRY_CAST(record_created_ts / 1000 AS BIGINT))
    END AS record_created_ts,

    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version AS _bronze_schema_version,
    _record_hash AS _bronze_record_hash
FROM trips_standardised;


In [ ]:
DESCRIBE trips_typed;


In [ ]:
CREATE OR REPLACE TEMP VIEW trips_candidate_ready AS
SELECT
    *,
    CASE
        WHEN request_ts IS NOT NULL
         AND driver_accept_ts IS NOT NULL
         AND driver_accept_ts >= request_ts
        THEN TIMESTAMPDIFF(SECOND, request_ts, driver_accept_ts)
    END AS response_seconds,

    CASE
        WHEN driver_accept_ts IS NOT NULL
         AND pickup_ts IS NOT NULL
         AND pickup_ts >= driver_accept_ts
        THEN TIMESTAMPDIFF(SECOND, driver_accept_ts, pickup_ts)
    END AS wait_seconds,

    CASE
        WHEN trip_status = 'completed'
         AND pickup_ts IS NOT NULL
         AND dropoff_ts IS NOT NULL
         AND dropoff_ts > pickup_ts
        THEN TIMESTAMPDIFF(SECOND, pickup_ts, dropoff_ts)
    END AS trip_duration_seconds,

    CASE
        WHEN trip_status IS NULL THEN NULL
        ELSE trip_status = 'completed'
    END AS is_completed,

    CASE
        WHEN trip_status IS NULL THEN NULL
        ELSE trip_status IN ('cancelled_by_rider', 'cancelled_by_driver')
    END AS is_cancelled,

    CASE
        WHEN trip_status IS NULL THEN NULL
        ELSE trip_status = 'unfulfilled'
    END AS is_unfulfilled,

    CASE
        WHEN surge_multiplier IS NOT NULL
        THEN surge_multiplier > CAST(1.00 AS DECIMAL(4,2))
    END AS is_surge_trip,

    CASE
        WHEN actual_distance_km IS NOT NULL
         AND estimated_distance_km IS NOT NULL
        THEN CAST(actual_distance_km - estimated_distance_km AS DECIMAL(8,2))
    END AS distance_variance_km,

    CASE
        WHEN trip_status = 'completed'
         AND final_fare_inr IS NOT NULL
         AND estimated_fare_inr IS NOT NULL
        THEN CAST(final_fare_inr - estimated_fare_inr AS DECIMAL(10,2))
    END AS fare_variance_inr

FROM trips_typed;


In [ ]:
SELECT
    trip_id,
    request_ts,
    driver_accept_ts,
    pickup_ts,
    dropoff_ts,
    trip_status,
    surge_multiplier,
    response_seconds,
    wait_seconds,
    trip_duration_seconds,
    is_completed,
    is_cancelled,
    is_unfulfilled,
    is_surge_trip,
    estimated_distance_km,
    actual_distance_km,
    distance_variance_km,
    estimated_fare_inr,
    final_fare_inr,
    fare_variance_inr
FROM trips_candidate_ready
LIMIT 20;


In [ ]:
CREATE OR REPLACE TABLE silver_trips_candidate
USING DELTA
AS
SELECT
    *,
    CURRENT_TIMESTAMP() AS _candidate_created_at,
    'trippulse_silver_candidate_v1.0' AS _candidate_schema_version
FROM trips_candidate_ready;


In [ ]:
SELECT *
FROM silver_trips_candidate
LIMIT 10;

# 7. Payments Candidate

**Bronze:** `bronze_trippulse_payments`  
**Candidate:** `silver_payments_candidate`  
**Grain:** one payment attempt

The approved rule is to preserve every attempt. We do **not** collapse payment attempts to trip grain. Group context is inspected with window functions without changing the physical row count.


In [ ]:
CREATE OR REPLACE TEMP VIEW payments_standardised AS
SELECT
    UPPER(TRIM(payment_id)) AS payment_id,
    UPPER(TRIM(trip_id)) AS trip_id,
    attempt_number,
    payment_ts,
    LOWER(TRIM(payment_method)) AS payment_method,
    LOWER(TRIM(payment_status)) AS payment_status,
    amount_inr,
    CASE
        WHEN failure_reason IS NULL THEN NULL
        ELSE LOWER(TRIM(failure_reason))
    END AS failure_reason,
    is_final_attempt,
    UPPER(TRIM(payment_reference)) AS payment_reference,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version,
    _record_hash
FROM bronze_trippulse_payments;


In [ ]:
CREATE OR REPLACE TEMP VIEW payments_typed AS
SELECT
    payment_id,
    trip_id,
    TRY_CAST(attempt_number AS INT) AS attempt_number,
    TRY_CAST(payment_ts AS TIMESTAMP) AS payment_ts,
    payment_method,
    payment_status,
    TRY_CAST(amount_inr AS DECIMAL(10,2)) AS amount_inr,
    failure_reason,
    TRY_CAST(is_final_attempt AS BOOLEAN) AS is_final_attempt,
    payment_reference,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version AS _bronze_schema_version,
    _record_hash AS _bronze_record_hash
FROM payments_standardised;


In [ ]:
DESCRIBE payments_typed;


In [ ]:
CREATE OR REPLACE TABLE silver_payments_candidate
USING DELTA
AS
SELECT
    *,
    CURRENT_TIMESTAMP() AS _candidate_created_at,
    'trippulse_silver_candidate_v1.0' AS _candidate_schema_version
FROM payments_typed;


In [ ]:
SELECT
    payment_id, trip_id, attempt_number, payment_ts,
    payment_method, payment_status, amount_inr,
    failure_reason, is_final_attempt, payment_reference,
    _source_file_name, _bronze_record_hash
FROM silver_payments_candidate
LIMIT 20;


### Payment-attempt group context check

This is an inspection query only. It does not create a new grain-changing table.


In [ ]:
SELECT
    trip_id,
    payment_id,
    attempt_number,
    payment_status,
    is_final_attempt,
    COUNT(*) OVER (PARTITION BY trip_id) AS attempts_for_trip
FROM silver_payments_candidate
ORDER BY trip_id, attempt_number
LIMIT 50;


# 8. Week-5 validation — Bronze = Candidate

The core Week-5 proof is row preservation. For the four available batch entities:

**Bronze count = Candidate count**

No `DISTINCT`, filter, deduplication or join is used to force equality.


In [ ]:
WITH counts AS (
    SELECT 'zones' AS entity,
           (SELECT COUNT(*) FROM bronze_trippulse_zones) AS bronze_rows,
           (SELECT COUNT(*) FROM silver_zones_candidate) AS candidate_rows
    UNION ALL
    SELECT 'drivers',
           (SELECT COUNT(*) FROM bronze_trippulse_drivers),
           (SELECT COUNT(*) FROM silver_drivers_candidate)
    UNION ALL
    SELECT 'trips',
           (SELECT COUNT(*) FROM bronze_trippulse_trips),
           (SELECT COUNT(*) FROM silver_trips_candidate)
    UNION ALL
    SELECT 'payments',
           (SELECT COUNT(*) FROM bronze_trippulse_payments),
           (SELECT COUNT(*) FROM silver_payments_candidate)
)
SELECT
    entity,
    bronze_rows,
    candidate_rows,
    candidate_rows - bronze_rows AS difference,
    CASE WHEN bronze_rows = candidate_rows THEN 'PASS' ELSE 'CHECK' END AS status
FROM counts
ORDER BY entity;


## 9. Row-level lineage proof

Counts alone are not sufficient. Compare the deterministic Bronze `_record_hash` with the retained `_bronze_record_hash`.


In [ ]:
SELECT
    'zones' AS entity,
    (SELECT COUNT(*) FROM (
        SELECT _record_hash FROM bronze_trippulse_zones
        EXCEPT ALL
        SELECT _bronze_record_hash FROM silver_zones_candidate
    )) AS bronze_hashes_missing_in_candidate,
    (SELECT COUNT(*) FROM (
        SELECT _bronze_record_hash FROM silver_zones_candidate
        EXCEPT ALL
        SELECT _record_hash FROM bronze_trippulse_zones
    )) AS unexpected_candidate_hashes

UNION ALL

SELECT
    'drivers',
    (SELECT COUNT(*) FROM (
        SELECT _record_hash FROM bronze_trippulse_drivers
        EXCEPT ALL
        SELECT _bronze_record_hash FROM silver_drivers_candidate
    )),
    (SELECT COUNT(*) FROM (
        SELECT _bronze_record_hash FROM silver_drivers_candidate
        EXCEPT ALL
        SELECT _record_hash FROM bronze_trippulse_drivers
    ))

UNION ALL

SELECT
    'trips',
    (SELECT COUNT(*) FROM (
        SELECT _record_hash FROM bronze_trippulse_trips
        EXCEPT ALL
        SELECT _bronze_record_hash FROM silver_trips_candidate
    )),
    (SELECT COUNT(*) FROM (
        SELECT _bronze_record_hash FROM silver_trips_candidate
        EXCEPT ALL
        SELECT _record_hash FROM bronze_trippulse_trips
    ))

UNION ALL

SELECT
    'payments',
    (SELECT COUNT(*) FROM (
        SELECT _record_hash FROM bronze_trippulse_payments
        EXCEPT ALL
        SELECT _bronze_record_hash FROM silver_payments_candidate
    )),
    (SELECT COUNT(*) FROM (
        SELECT _bronze_record_hash FROM silver_payments_candidate
        EXCEPT ALL
        SELECT _record_hash FROM bronze_trippulse_payments
    ))
ORDER BY entity;


## 10. Safe-cast / conversion-failure inspection

A failed conversion does **not** delete a row. Week 6 decides DQ outcomes.

The checks below count only non-null Bronze values that become null after the approved conversion.


In [ ]:
SELECT
    'zones.effective_from' AS field,
    SUM(CASE WHEN effective_from IS NOT NULL
              AND TRY_CAST(effective_from AS DATE) IS NULL
             THEN 1 ELSE 0 END) AS parse_failures
FROM bronze_trippulse_zones
UNION ALL
SELECT
    'drivers.onboard_date',
    SUM(CASE WHEN onboard_date IS NOT NULL
              AND TRY_CAST(onboard_date AS DATE) IS NULL
             THEN 1 ELSE 0 END)
FROM bronze_trippulse_drivers
UNION ALL
SELECT
    'drivers.last_status_update_ts',
    SUM(CASE WHEN last_status_update_ts IS NOT NULL
              AND TRY_CAST(last_status_update_ts AS TIMESTAMP) IS NULL
             THEN 1 ELSE 0 END)
FROM bronze_trippulse_drivers
UNION ALL
SELECT
    'drivers.rating',
    SUM(CASE WHEN rating IS NOT NULL
              AND TRY_CAST(rating AS DECIMAL(3,2)) IS NULL
             THEN 1 ELSE 0 END)
FROM bronze_trippulse_drivers
UNION ALL
SELECT
    'payments.payment_ts',
    SUM(CASE WHEN payment_ts IS NOT NULL
              AND TRY_CAST(payment_ts AS TIMESTAMP) IS NULL
             THEN 1 ELSE 0 END)
FROM bronze_trippulse_payments
UNION ALL
SELECT
    'payments.amount_inr',
    SUM(CASE WHEN amount_inr IS NOT NULL
              AND TRY_CAST(amount_inr AS DECIMAL(10,2)) IS NULL
             THEN 1 ELSE 0 END)
FROM bronze_trippulse_payments
ORDER BY field;


### Trip timestamp conversion inspection

Trip Bronze stores lifecycle timestamps as epoch nanoseconds. The conversion below checks whether a non-null source value produced a null Candidate timestamp.


In [ ]:
SELECT
    SUM(CASE WHEN request_ts IS NOT NULL
              AND request_ts IS NOT NULL
              AND request_ts_typed IS NULL
             THEN 1 ELSE 0 END) AS request_ts_conversion_failures,
    SUM(CASE WHEN driver_accept_ts IS NOT NULL
              AND driver_accept_ts_typed IS NULL
             THEN 1 ELSE 0 END) AS driver_accept_ts_conversion_failures,
    SUM(CASE WHEN pickup_ts IS NOT NULL
              AND pickup_ts_typed IS NULL
             THEN 1 ELSE 0 END) AS pickup_ts_conversion_failures,
    SUM(CASE WHEN dropoff_ts IS NOT NULL
              AND dropoff_ts_typed IS NULL
             THEN 1 ELSE 0 END) AS dropoff_ts_conversion_failures,
    SUM(CASE WHEN cancel_ts IS NOT NULL
              AND cancel_ts_typed IS NULL
             THEN 1 ELSE 0 END) AS cancel_ts_conversion_failures,
    SUM(CASE WHEN record_created_ts IS NOT NULL
              AND record_created_ts_typed IS NULL
             THEN 1 ELSE 0 END) AS record_created_ts_conversion_failures
FROM (
    SELECT
        request_ts,
        CASE WHEN request_ts IS NOT NULL
             THEN timestamp_micros(TRY_CAST(request_ts / 1000 AS BIGINT)) END AS request_ts_typed,
        driver_accept_ts,
        CASE WHEN driver_accept_ts IS NOT NULL
             THEN timestamp_micros(TRY_CAST(driver_accept_ts / 1000 AS BIGINT)) END AS driver_accept_ts_typed,
        pickup_ts,
        CASE WHEN pickup_ts IS NOT NULL
             THEN timestamp_micros(TRY_CAST(pickup_ts / 1000 AS BIGINT)) END AS pickup_ts_typed,
        dropoff_ts,
        CASE WHEN dropoff_ts IS NOT NULL
             THEN timestamp_micros(TRY_CAST(dropoff_ts / 1000 AS BIGINT)) END AS dropoff_ts_typed,
        cancel_ts,
        CASE WHEN cancel_ts IS NOT NULL
             THEN timestamp_micros(TRY_CAST(cancel_ts / 1000 AS BIGINT)) END AS cancel_ts_typed,
        record_created_ts,
        CASE WHEN record_created_ts IS NOT NULL
             THEN timestamp_micros(TRY_CAST(record_created_ts / 1000 AS BIGINT)) END AS record_created_ts_typed
    FROM bronze_trippulse_trips
) t;


## 11. Trip derivation spot-check

Use one or more real rows from the Candidate output and manually verify the arithmetic. The query intentionally shows inputs beside derived values.


In [ ]:
SELECT
    trip_id,
    request_ts,
    driver_accept_ts,
    pickup_ts,
    dropoff_ts,
    trip_status,
    response_seconds,
    wait_seconds,
    trip_duration_seconds,
    is_completed,
    is_cancelled,
    is_unfulfilled,
    is_surge_trip,
    estimated_distance_km,
    actual_distance_km,
    distance_variance_km,
    estimated_fare_inr,
    final_fare_inr,
    fare_variance_inr
FROM silver_trips_candidate
WHERE response_seconds IS NOT NULL
   OR trip_duration_seconds IS NOT NULL
   OR distance_variance_km IS NOT NULL
   OR fare_variance_inr IS NOT NULL
LIMIT 10;


## 12. Lifecycle / null-behaviour spot checks

These checks confirm that Week 5 did not invent timestamps or fares for non-applicable lifecycle states.


In [ ]:
SELECT
    trip_status,
    COUNT(*) AS rows,
    SUM(CASE WHEN is_completed THEN 1 ELSE 0 END) AS completed_rows,
    SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) AS cancelled_rows,
    SUM(CASE WHEN is_unfulfilled THEN 1 ELSE 0 END) AS unfulfilled_rows,
    SUM(CASE WHEN trip_duration_seconds IS NULL THEN 1 ELSE 0 END) AS null_duration_rows,
    SUM(CASE WHEN final_fare_inr IS NULL THEN 1 ELSE 0 END) AS null_final_fare_rows
FROM silver_trips_candidate
GROUP BY trip_status
ORDER BY trip_status;


## 13. Payment grain proof

One Candidate row must remain one payment attempt. This check validates the declared business key shape without collapsing the table.


In [ ]:
SELECT
    COUNT(*) AS candidate_payment_rows,
    COUNT(DISTINCT payment_id) AS distinct_payment_ids,
    COUNT(DISTINCT CONCAT(trip_id, '||', CAST(attempt_number AS STRING))) AS distinct_trip_attempt_keys,
    SUM(CASE WHEN is_final_attempt THEN 1 ELSE 0 END) AS final_attempt_rows
FROM silver_payments_candidate;


In [ ]:
SELECT
    trip_id,
    COUNT(*) AS attempt_rows,
    COUNT(DISTINCT attempt_number) AS distinct_attempt_numbers,
    SUM(CASE WHEN is_final_attempt THEN 1 ELSE 0 END) AS final_flags
FROM silver_payments_candidate
GROUP BY trip_id
HAVING COUNT(*) <> COUNT(DISTINCT payment_id)
    OR SUM(CASE WHEN is_final_attempt THEN 1 ELSE 0 END) <> 1
LIMIT 20;


## 14. Candidate schemas and Delta format

The Candidate outputs must be persistent Delta tables with the approved typed columns and lineage.


In [ ]:
SELECT table_name, data_source_format
FROM system.information_schema.tables
WHERE table_catalog = current_catalog()
  AND table_schema = current_schema()
  AND table_name IN (
      'silver_zones_candidate',
      'silver_drivers_candidate',
      'silver_trips_candidate',
      'silver_payments_candidate',
      'silver_ride_request_events_candidate'
  )
ORDER BY table_name;


## 15. Before/after record evidence

Capture one real record for the Week-5 evidence folder. The query below makes the transformation visible without using a PageLoop sample.


In [ ]:
SELECT
    b.trip_id AS bronze_trip_id,
    b.request_ts AS bronze_request_ts_nanos,
    b.trip_status AS bronze_trip_status,
    b.surge_multiplier AS bronze_surge_multiplier,
    c.trip_id AS candidate_trip_id,
    c.request_ts AS candidate_request_ts,
    c.trip_status AS candidate_trip_status,
    c.surge_multiplier AS candidate_surge_multiplier,
    c.response_seconds,
    c.is_completed,
    b._record_hash AS bronze_record_hash,
    c._bronze_record_hash AS candidate_bronze_record_hash
FROM bronze_trippulse_trips b
JOIN silver_trips_candidate c
  ON b._record_hash = c._bronze_record_hash
LIMIT 1;


## 16. Controlled rerun test

The Candidate writes use `CREATE OR REPLACE TABLE`, matching the snapshot-style method demonstrated in the conversion guide.

1. Record the current counts.
2. Rerun the four Candidate table creation cells.
3. Rerun the B = C and lineage checks.
4. Confirm counts and lineage remain stable.

Do not claim rerun safety merely because the SQL completed; the post-rerun reconciliation is the proof.


In [ ]:
SELECT
    'zones' AS entity, COUNT(*) AS candidate_rows FROM silver_zones_candidate
UNION ALL
SELECT 'drivers', COUNT(*) FROM silver_drivers_candidate
UNION ALL
SELECT 'trips', COUNT(*) FROM silver_trips_candidate
UNION ALL
SELECT 'payments', COUNT(*) FROM silver_payments_candidate
ORDER BY entity;


## 17. Week-5 close

### Completed in this notebook
- Four available TripPulse Bronze batch inputs transformed into project-specific Silver Candidate Delta tables.
- Approved standardisation and typing applied.
- Approved trip derivations implemented with eligibility conditions.
- Payment attempt grain preserved.
- Bronze lineage retained.
- Count, lineage, conversion, lifecycle and derivation validation included.
- Controlled replace/rerun proof included.

### Week-6 boundary
Do not add Trusted Silver, quarantine routing, Gold, Power BI or streaming implementation here. Candidate remains explicitly untrusted until the approved Week-6 DQ rules are applied.


## Evidence checklist

- [ ] `notebooks/03_silver_transformations.ipynb` committed at the exact plural path.
- [ ] Four available Candidate tables exist as Delta and the event dependency is documented.
- [ ] Bronze = Candidate counts pass for zones, drivers, trips and payments.
- [ ] Bronze/Candidate record-hash reconciliation passes.
- [ ] Candidate schemas show the approved target types and retained lineage.
- [ ] One real Bronze-vs-Candidate record is captured.
- [ ] Trip derivation spot checks are captured.
- [ ] Null/lifecycle behaviour is captured.
- [ ] Payment attempt grain is captured.
- [ ] Controlled rerun counts are stable.
- [ ] `docs/data_dictionary.md` and `docs/pipeline_walkthrough.md` are updated.
- [ ] `weekly_logs/week05_log.md` records actual work, decisions, blockers, evidence and AI use.
- [ ] No fabricated PASS result, count, screenshot or event-table output is added.
